# RetNet-300M Coding & Reasoning Agent

Trains **RetNet 2.0 (~310M params)** with **Delta-Rule Memory (`β < 2`) + Short Convolution (`K=4`) + Calculator-in-the-Loop**
strictly on **high-quality synthetic textbooks, code, and step-by-step reasoning** (`nampdn-ai/tiny-series` + `MetaMathQA` + `GSM8K` + `Magicoder`).

### Setup (One-Time)
- In Google Drive, create **`My Drive/retnet-300m/`** and upload **`retnet-300m-colab.zip`** into it.
- Click **Runtime → Run all**.

### Resuming
- After a disconnect, click **Runtime → Run all** again. Checkpoints are saved as versioned step files (`ckpt_step_XXXXXXX.pt`) every 250 steps / 15 minutes (and on stop signals); the loader skips a checkpoint whose upload was cut off and resumes from the previous one.

In [ ]:
# 1. Check GPU (optional: CPU is enough for data prep)
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
# A CPU runtime is fine for data prep (cells 2-4); training (cell 6) needs a GPU.
if torch.cuda.is_available():
    GPU = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"{GPU} ({VRAM_GB:.1f} GB) | capability {torch.cuda.get_device_capability(0)} | torch {torch.__version__}")
else:
    GPU = None
    print(f"CPU runtime (torch {torch.__version__}): data prep will run; training stops at cell 6.")

In [ ]:
# 2. Mount Drive, find the project folder, unpack code
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib, shutil, subprocess, sys, json

# Optional: paste your Hugging Face token once (or set HF_TOKEN in Colab Secrets).
# It is saved to DRIVE / '.hf_token' so later sessions reuse it.
HF_TOKEN = ""  # e.g. "hf_..." (after clicking Agree on nampdn-ai/tiny-* pages; leave empty for automatic ungated fallbacks)
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN") or ""
except Exception:
    pass

# Search MyDrive, Shareddrives, and .shortcut-targets-by-id for 'retnet-300m'
FOLDER_NAME = "retnet-300m"
MOUNT = pathlib.Path("/content/drive")
candidates = [MOUNT / "MyDrive" / FOLDER_NAME]
if (MOUNT / "Shareddrives").is_dir():
    candidates += [d / FOLDER_NAME for d in sorted((MOUNT / "Shareddrives").iterdir()) if d.is_dir()]
if (MOUNT / ".shortcut-targets-by-id").is_dir():
    candidates += [d / FOLDER_NAME for d in sorted((MOUNT / ".shortcut-targets-by-id").iterdir()) if d.is_dir()]

DRIVE = next((c for c in candidates if c.is_dir()), MOUNT / "MyDrive" / FOLDER_NAME)
BUNDLE = DRIVE / "retnet-300m-colab.zip"
assert BUNDLE.is_file(), (
    f"Could not find {BUNDLE}.\n"
    "Upload dist/retnet-300m-colab.zip to MyDrive/retnet-300m/."
)

CODE = pathlib.Path("/content/retnet-300m")
DATA_LOCAL = pathlib.Path("/content/data/tinymix")
DATA_DRIVE = DRIVE / "data" / "tinymix"
SHARD_CACHE = DRIVE / "data" / "tinymix_shards"
RUNS = DRIVE / "runs"
PY = sys.executable
RUNS.mkdir(parents=True, exist_ok=True)

def run(cmd):
    """Run a shell command, stream its output, and STOP the notebook on failure."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")

os.chdir("/content")
shutil.rmtree(CODE, ignore_errors=True)
run(f"unzip -oq '{BUNDLE}' -d '{CODE}'")
os.chdir(CODE)

# Refuse to run an outdated bundle: re-upload dist/retnet-300m-colab.zip after every code change.
REQUIRED_VERSION = (0, 5, 0)
import re as _re
_m = _re.search(r'__version__ = "([^"]+)"', (CODE / "retnet" / "__init__.py").read_text())
BUNDLE_VERSION = tuple(int(x) for x in _m.group(1).split(".")) if _m else (0, 0, 0)
_info = (CODE / "BUILD_INFO.txt").read_text().strip() if (CODE / "BUILD_INFO.txt").is_file() else "no BUILD_INFO.txt"
print(f"Code bundle: {_info}")
assert BUNDLE_VERSION >= REQUIRED_VERSION, (
    f"{BUNDLE} is outdated (version {'.'.join(map(str, BUNDLE_VERSION))}, need {'.'.join(map(str, REQUIRED_VERSION))}).\n"
    "Rebuild it locally with `bash scripts/make_colab_bundle.sh` and upload dist/retnet-300m-colab.zip to the Drive folder, replacing the old one."
)
run(f"{PY} -m pip -q install tokenizers datasets")

sys.path.insert(0, str(CODE / "scripts"))
sys.path.insert(0, str(CODE))
from colab_drive import resolve_shared_drive_dir, is_valid_token_cache, restore_data_from_drive, sync_data_to_drive
DRIVE = resolve_shared_drive_dir(FOLDER_NAME, MOUNT)
hf_file = DRIVE / ".hf_token"
if HF_TOKEN:
    hf_file.write_text(HF_TOKEN.strip())
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("Saved HF_TOKEN to Drive (.hf_token) for later sessions.")
elif hf_file.is_file():
    os.environ["HF_TOKEN"] = hf_file.read_text().strip()
    print("Loaded shared HF_TOKEN from Drive (.hf_token).")
print(f"Drive folder active & writable at: {DRIVE}")

In [ ]:
# 3. Sanity check (~30s): retention modes agree, delta memory overwrites, calculator & checkpoint recovery work
run(f"{PY} -m pytest -q tests")

In [ ]:
# 4. Tiny-Series + Reasoning Data (1B tokens; cached & verified on Drive)
DATA_TOKENS = 1e9
if restore_data_from_drive(DATA_DRIVE, DATA_LOCAL):
    print("Verified & restored tokenized dataset from Drive to local SSD.")
else:
    print("Preparing Tiny-Series + Reasoning dataset. Progress is saved to the Drive shard cache every 30 s,\n"
          "so after a disconnect, Run All continues from where it stopped.")
    run(f"{PY} -u scripts/prepare_mix.py --out '{DATA_LOCAL}' --cache_dir '{SHARD_CACHE}' --total_tokens {DATA_TOKENS:.0f}")
    print("Syncing verified dataset to Drive...")
    sync_data_to_drive(DATA_LOCAL, DATA_DRIVE)
    shutil.rmtree(SHARD_CACHE, ignore_errors=True)  # free shard cache once full dataset is verified on Drive

meta = json.load(open(DATA_LOCAL / "meta.json"))
print(f"Total: {meta['train_tokens'] / 1e6:.1f}M train tokens, {meta['val_tokens'] / 1e6:.2f}M val tokens (vocab {meta['vocab_size']})")
for name, info in meta["sources"].items():
    print(f"  {name:22s} {info['train_tokens'] / 1e6:6.1f}M train  ({info['passes']} pass, "
          f"{info.get('source') or 'cached'})")

## Configure RetNet-300M Training

* **`retnet-300m`**: `d_model=1024, n_layers=22, n_heads=16` (~277M non-embedding / **~310M total parameters**) with
  `--memory delta --beta_max 2 --short_conv 4` and calculator-in-the-loop.
* Automatically measures the largest micro-batch that fits the current GPU (`T4`, `L4`, or `A100`), enabling
  per-layer activation checkpointing automatically on 16 GB T4 GPUs if needed, while keeping the exact same
  effective batch size (`32 sequences × 1024 tokens = 32,768 tokens/step`) across GPU types.

In [ ]:
# 5. Configure run(s)
import shlex
from retnet.checkpoint import load_latest_checkpoint

RUN_ORDER = ["retnet-300m"]  # Add "retnet-125m" if you also want to train a smaller comparison model
TRAIN_TOKENS = 5e8            # Increase to 1e9 for a full epoch over 1B tokens
COMPILE = True

VARIANTS = {
    "retnet-300m": "--preset 300m --memory delta --beta_max 2 --short_conv 4 --lr 4e-4",
    "retnet-300m-deep": "--preset 300m-deep --memory delta --beta_max 2 --short_conv 4 --lr 4e-4",
    "retnet-125m": "--preset 125m --memory delta --beta_max 2 --short_conv 4 --lr 6e-4",
}
SEQ, SEQS_PER_STEP = 1024, 32  # 32,768 tokens per optimizer step
STEPS = int(TRAIN_TOKENS // (SEQ * SEQS_PER_STEP))
SAMPLE = shlex.quote("Question: A baker bakes 7 trays of 24 muffins and sells 150. How many are left?\\nAnswer:")

def train_args(name):
    return (
        f"{VARIANTS[name]} --seq_len {SEQ} --batch_size 0 --seqs_per_step {SEQS_PER_STEP} "
        f"--max_steps {STEPS} --warmup_steps 1000 --eval_every 500 --save_every 250 "
        f"--save_every_minutes 15 --keep_ckpts 2 --eval_batches 20 "
        f"--train_mode chunkwise --chunk_size 128 --sample_prompt {SAMPLE}"
        + (" --compile" if COMPILE else "")
    )

def current_step(name):
    ck, p = load_latest_checkpoint(RUNS / name, "cpu")
    return (int(ck.get("step", 0)), p.name) if ck is not None else (0, None)

print(f"Target: {STEPS} steps x {SEQ * SEQS_PER_STEP} tokens/step = {STEPS * SEQ * SEQS_PER_STEP / 1e6:.0f}M tokens on {GPU or 'CPU (data prep only)'}")
for name in RUN_ORDER:
    st, fn = current_step(name)
    status = f"DONE ({st}/{STEPS})" if st >= STEPS else (f"RESUMABLE at step {st}/{STEPS} ({fn})" if st > 0 else "NOT STARTED")
    print(f"  {name:18s} : {status}")

In [ ]:
# 6. Train & Evaluate (safe to stop at any time; Run All resumes seamlessly)
assert GPU, "Data is ready. Training needs a GPU: Runtime -> Change runtime type -> T4 GPU, then Run All again."
for name in RUN_ORDER:
    out = RUNS / name
    st, _ = current_step(name)
    print(f"\n==================== {name} (step {st}/{STEPS})")
    if st < STEPS:
        run(f"{PY} -u scripts/train.py --name {name} --data '{DATA_LOCAL}' --out_root '{RUNS}' "
            f"--resume --dtype auto {train_args(name)}")
    st_after, _ = current_step(name)
    if st_after < STEPS:
        print(f"Run {name} paused at step {st_after}/{STEPS}; re-run the cell to continue.")
        continue
    evals = [
        ("eval_mc.json", f"--data '{DATA_LOCAL}' --tasks val_by_source,arc_easy,arc_challenge,hellaswag,mbpp"),
        ("eval.json", "--tasks gsm8k"),
        ("eval_nocalc.json", "--tasks gsm8k --no_calculator"),
    ]
    for fname, args in evals:
        if (out / fname).is_file():
            print(f"{fname}: already done")
        else:
            run(f"{PY} -u scripts/eval.py --run '{out}' {args} --out '{out / fname}'")

In [ ]:
# 7. Plot Learning Curves & Evaluation Summary
import matplotlib.pyplot as plt

def load_json(path):
    try:
        return json.load(open(path))
    except (OSError, ValueError):
        return {}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name in RUN_ORDER:
    d = RUNS / name
    if not (d / "log.jsonl").is_file():
        continue
    recs = [json.loads(l) for l in open(d / "log.jsonl") if l.strip()]
    ev = [r for r in recs if "val_loss" in r]
    if recs:
        ax[0].plot([r["tokens"] / 1e6 for r in recs], [r["train_loss"] for r in recs], alpha=0.7, label=name)
    if ev:
        ax[1].plot([r["tokens"] / 1e6 for r in ev], [r["val_loss"] for r in ev], "o-", label=name)
    mc, g, gn = load_json(d / "eval_mc.json"), load_json(d / "eval.json"), load_json(d / "eval_nocalc.json")
    if mc or g:
        print(f"=== {name} ===")
        if "val_by_source" in mc:
            print("  Held-out val loss by source:", mc["val_by_source"])
        for k in ("arc_easy", "arc_challenge", "hellaswag", "mbpp"):
            if k in mc:
                print(f"  {k:15s}: {mc[k]}")
        if "gsm8k" in g:
            print(f"  gsm8k (calc)   : {100*g['gsm8k']['acc']:.1f}% +/- {100*g['gsm8k']['stderr']:.1f}% ({g['gsm8k']['calculator_calls_per_problem']:.2f} calls/prob)")
        if "gsm8k" in gn:
            print(f"  gsm8k (no calc): {100*gn['gsm8k']['acc']:.1f}% +/- {100*gn['gsm8k']['stderr']:.1f}%")

for a_plot, title in zip(ax, ["Train Loss", "Validation Loss"]):
    a_plot.set_xlabel("Tokens (M)"); a_plot.set_title(title); a_plot.set_yscale("log"); a_plot.grid(alpha=0.3); a_plot.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 8. Test the Coding & Reasoning Agent (greedy decoding + calculator-in-the-loop)
NAME = "retnet-300m"
MATH_Q = "A baker makes 24 muffins per tray and bakes 7 trays. He sells 150 muffins. How many are left?"
CODE_Q = "Write a Python function `binary_search(arr, target)` that returns the index of `target` in sorted list `arr`, or -1 if not found."

run(f"{PY} scripts/chat.py --run '{RUNS / NAME}' --prompt " + shlex.quote(f"Question: {MATH_Q}\\nAnswer:"))
run(f"{PY} scripts/chat.py --run '{RUNS / NAME}' --code " + shlex.quote(CODE_Q))